# 05 — One-stage anchor-free: FCOS

In [ ]:
import subprocess
import sys
from pathlib import Path

if Path("../src/vod").exists():
    sys.path.insert(0, "../src")
else:
    repo = Path("/tmp/repo")  # outside /kaggle/working so it is not saved as output
    if not repo.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/mohinska/ai_hw2_MultipleObjectVideoDetection", str(repo)], check=True)
    sys.path.insert(0, str(repo / "src"))

In [ ]:
!pip install -q faster-coco-eval

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from vod import CLASSES
from vod.data import find_data_root, load_split
from vod.protocol import PROTOCOL
from vod.submission import write_submission
from vod.tv import (comparison_tables, completed_runs, fcos_assignment_stats, load_run_model, predict_test,
                    run_all)

DATA = find_data_root()
WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("../outputs")

## Data (same protocol as every detector)

In [ ]:
train_frames, train_targets = load_split(DATA / "train", step=PROTOCOL["train_step"])
val_frames, val_targets = load_split(DATA / "val", step=1)
VAL_GT = DATA / "val" / "annotations_coco.json"
print(len(train_frames), "train frames,", len(val_frames), "val frames")

## How FCOS assigns locations to objects
There are no anchors. Every location of the feature maps (strides 8-128) is a candidate; it is a positive for a box
if it lies inside the box **and** within `radius x stride` of the box centre (centre sampling). The radius is the
main knob: a small radius leaves tiny objects with no positive location, a large one gives them more training
signal but also pulls in locations that belong to a neighbour in crowded scenes. Below: positive locations at
the finest level (stride 8) per training box, by object size at network input, for the radii we search.

In [ ]:
RADII = [0.5, 1.5, 2.5]  # 1.5 is the torchvision default
for radius in RADII:
    print(f"centre sampling radius {radius}")
    display(fcos_assignment_stats(train_frames, train_targets, PROTOCOL["imgsz"], radius))

## Experiments
Each experiment changes one thing relative to `fcos_default` (pretrained 4-conv tower kept, only the last
classification conv replaced). All of `RUN` are trained one after another (~1.5 h each on a T4); every finished
run is saved right away, so a crash keeps earlier runs.

In [ ]:
BASE = dict(arch="fcos", lr=0.0025)
EXPERIMENTS = {
    "fcos_default": BASE,
    # assignment: fewer / more positive locations per object
    "fcos_radius0.5": {**BASE, "center_sampling_radius": 0.5},
    "fcos_radius2.5": {**BASE, "center_sampling_radius": 2.5},
    # loss modification: double weight of the GIoU box regression loss (AP75 is the weak point)
    "fcos_boxloss2": {**BASE, "loss_weights": {"bbox_regression": 2.0}},
}
RUN = list(EXPERIMENTS)

## Train and evaluate (last epoch, every val frame)
Runs attached as input are not retrained; runs evaluated with older code (best epoch) are re-evaluated
from their last.pt so that every number comes from the same procedure.

In [ ]:
run_all(RUN, EXPERIMENTS, PROTOCOL, train_frames, train_targets, val_frames, val_targets, VAL_GT, WORK)

## Comparison with every finished run
Attach the Faster R-CNN (03) and RetinaNet (04) outputs as input to compare with the anchor-based detectors.

In [ ]:
runs = completed_runs((WORK, Path("/kaggle/input")))
main, per_class, per_clip = comparison_tables(runs, CLASSES)
display(main)
display(per_class)
per_clip  # mAP of each val clip: is a difference consistent across scenes?

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for n, m in runs.items():
    if m.get("history") is not None:
        ax.plot(m["history"]["epoch"], m["history"]["mAP"], marker="o", label=n)
ax.set(xlabel="epoch", ylabel="val mAP (every 10th frame)", title="Validation mAP during training")
ax.legend(fontsize=7)
plt.tight_layout()
plt.savefig(WORK / "map_curves.png", dpi=110)

## Test predictions with the best FCOS run (by val mAP)

In [ ]:
BEST = main[main.index.isin(EXPERIMENTS)]["mAP"].idxmax()
cfg = {**PROTOCOL, **EXPERIMENTS[BEST]}
model = load_run_model(BEST, cfg, (WORK, Path("/kaggle/input")))
print("best:", BEST)
sub = write_submission(predict_test(model, cfg, DATA), DATA / "test_images.csv", WORK / f"submission_{BEST}.csv")
sub.head()